# 02 · Preparação dos dados (v2)
**CRISP-DM:** Preparação dos Dados

1. Consolidação das chaves repetidas;
2. **Janelas de 1 h** por recurso (base do perfil normal);
3. **Períodos de 15 e 30 min** alinhados em agosto (para calibrar os limites dessas faixas contra o perfil de 1 h);
4. Mesmo período 7 dias antes (para o baseline semana anterior).

| Atributo (decisão) | Definição |
|---|---|
| `log_vol` | log(1 + transações), **na escala horária** (× 60 / minutos do período) |
| `tx_4xx` | transações 4xx / total (erros de negócio) |
| `tx_5xx` | transações 5xx / total (erros de sistema) |
| `log_lat_p95` | log(1 + **p95 da latência**): percentil 95 ponderado pelo volume das latências médias por minuto × cliente das respostas 2xx |

**Sobre o p95.** Como cada linha já é a média de um minuto × cliente, o p95 é o percentil **dessas médias** (ponderado pelo número de transações), e não das requisições individuais. Ele é consistente entre janelas e sensível a degradações que atingem só parte dos minutos ou dos clientes; o p95 real por requisição tende a ser maior.

Contexto: hora (Brasília) e tipo de dia (**útil / não útil**; feriado = não útil).

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v2) carregado · /home/wsl/gh/mestrado/claude9/v2


In [2]:
brutos = consolidar(carregar())
print(f"registros consolidados: {len(brutos):,}")
brutos.to_parquet(RESULTADOS / "02_brutos_consolidados.parquet", index=False)

registros consolidados: 2,283,149


In [3]:
jan1h = agregar(brutos, "1h")
jan1h.to_parquet(RESULTADOS / "02_janelas_60min.parquet", index=False)
for L in (15, 30):
    pL = agregar(brutos, f"{L}min")
    pL.to_parquet(RESULTADOS / f"02_janelas_{L}min.parquet", index=False)
    print(f"{L} min: {len(pL):,} períodos")
print(f"1 h: {len(jan1h):,} janelas")

15 min: 42,783 períodos


30 min: 21,391 períodos
1 h: 10,695 janelas


## Verificação: agregação por janela × agregação de período arbitrário
As duas rotinas (vetorizada, usada na calibração; e por período, usada nas consultas) devem dar o mesmo resultado para o mesmo intervalo.

In [4]:
B = Brutos(brutos)
amostra = jan1h.dropna(subset=["lat_p95"]).sample(200, random_state=0)
dif = []
for row in amostra.itertuples():
    f = B.fatia(row.recurso, row.janela, row.janela + pd.Timedelta("1h"))
    o = agregar_periodo(f, row.janela, row.janela + pd.Timedelta("1h"))
    dif.append([abs(o[a] - getattr(row, a)) for a in ATRIBUTOS])
print("maior diferença por atributo:", dict(zip(ATRIBUTOS, np.nanmax(dif, axis=0).round(9))))

maior diferença por atributo: {'log_vol': np.float64(0.0), 'tx_4xx': np.float64(0.0), 'tx_5xx': np.float64(0.0), 'log_lat_p95': np.float64(0.0)}


## Latência p95 × latência média

In [5]:
ok2 = brutos[brutos["status"] // 100 == 2].assign(janela=lambda d: d["ts"].dt.floor("1h"), tt=lambda d: d["qtd"] * d["tmp"])
media2 = ok2.groupby(["recurso", "janela"])[["tt", "qtd"]].sum().eval("tt / qtd").rename("lat_media")
comp = jan1h.set_index(["recurso", "janela"])[["lat_p95"]].join(media2)
print("razão p95 / média das respostas 2xx (mediana):", round(float((comp["lat_p95"] / comp["lat_media"]).median()), 2))
jan1h[ATRIBUTOS + ["vol", "lat_p95", "n_cli"]].describe().T.round(3)

razão p95 / média das respostas 2xx (mediana): 1.43


,count,mean,std,min,25%,50%,75%,max
log_vol,10695.0,7.140,1.349,2.485,6.301,7.039,7.970,11.700
tx_4xx,10695.0,0.281,0.377,0.000,0.001,0.015,0.571,1.000
tx_5xx,10695.0,0.003,0.023,0.000,0.000,0.000,0.000,0.821
log_lat_p95,10187.0,4.579,0.704,3.258,4.025,4.382,5.094,8.125
vol,10695.0,3355.371,8348.082,11.000,544.000,1139.000,2890.500,120553.000
lat_p95,10187.0,131.937,165.833,25.000,55.000,79.000,162.000,3378.000
n_cli,10695.0,24.719,38.499,1.000,5.000,12.000,30.000,494.000


## Amostras por contexto no treino (1 h)

In [6]:
tr = recortar(jan1h, TREINO)
tr.groupby(["recurso", "tipo_dia", "hora"]).size().groupby("tipo_dia").describe()[["min", "50%", "max"]]

,min,50%,max
tipo_dia,,,
nao_util,10.0,10.0,10.0
util,21.0,21.0,21.0
